# 레슨 09 — 속도 제한, 재시도, 로그

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kevin-innovation/jupyter-lecture/blob/main/python-web-automation/lectures/09/[학생용] 레슨 09 — 속도 제한, 재시도, 로그.ipynb)

이 노트북은 읽기와 따라하기용 강의 노트북이다. 자동화가 실패 응답을 만났을 때 무한 반복하지 않고, 요청 간격과 재시도 정책, 로그를 남기는 운영형 수집기을 안전한 합성 fixture로 연습한다.

## 학습 목표

1. 재시도해야 하는 상태 코드와 즉시 중단할 상태 코드를 구분한다.
2. 지수 backoff와 최대 재시도 횟수를 코드로 표현한다.
3. robots 샘플 규칙과 요청 간격을 자동화 흐름에 반영한다.
4. 시도별 로그와 최종 요약을 CSV/JSON으로 저장한다.
5. 운영 중 실패를 숨기지 않고 보고서로 남긴다.

---

## 1. 수업 맥락과 안전 기준

실제 운영 자동화는 한 번에 성공하지 않는다. 네트워크 지연, 임시 장애, 속도 제한이 섞인다. 이 레슨은 외부 사이트에 반복 요청을 보내지 않고 계획된 응답 fixture로 실패와 회복을 안전하게 연습한다.

자동화는 빠르게 반복하는 도구이기 때문에 실패했을 때 더 위험해질 수 있다. 그래서 이번 레슨에서는 모든 입력을 수업용 파일로 고정하고, 결과를 저장하기 전에 검증하거나 로그를 남기는 과정을 코드에 포함한다. 이 습관은 실제 사이트를 대상으로 할 때 요청량을 줄이고, 오류를 빨리 발견하게 만든다.

## 2. 환경 셀


In [ ]:
import os
import re
import csv
import json
import time
import sqlite3
import logging
from pathlib import Path
from urllib.parse import urljoin, urlparse

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/09/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def load_csv(filename):
    text = load_text(filename)
    return list(csv.DictReader(text.splitlines()))

def load_json(filename):
    return json.loads(load_text(filename))

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', str(text)) or '0')

def write_csv(path, rows, fieldnames=None):
    rows = list(rows)
    if fieldnames is None:
        fieldnames = list(rows[0].keys()) if rows else []
    with open(path, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    return path

print('colab:', IS_COLAB)
print('data base:', DATA_BASE)

class PlannedFetcher:
    def __init__(self, response_rows):
        self.plan = {}
        self.cursor = {}
        for row in response_rows:
            self.plan.setdefault(row['endpoint_id'], []).append(row)
        for endpoint_id in self.plan:
            self.plan[endpoint_id].sort(key=lambda row: int(row['attempt']))
            self.cursor[endpoint_id] = 0
    def get(self, endpoint_id):
        rows = self.plan.get(endpoint_id, [])
        if not rows:
            return {'endpoint_id': endpoint_id, 'attempt': 1, 'status_code': 404, 'body': 'missing-plan'}
        idx = min(self.cursor[endpoint_id], len(rows) - 1)
        self.cursor[endpoint_id] += 1
        row = dict(rows[idx])
        row['status_code'] = int(row['status_code'])
        row['attempt'] = int(row['attempt'])
        return row

def should_retry(status_code):
    return int(status_code) in {408, 429, 500, 502, 503, 504}

def backoff_delay(attempt, base=1, cap=8):
    return min(cap, base * (2 ** max(0, attempt - 1)))

def parse_disallow(text):
    blocked = []
    for line in text.splitlines():
        line = line.strip()
        if line.lower().startswith('disallow:'):
            path = line.split(':', 1)[1].strip()
            if path:
                blocked.append(path)
    return blocked



---

## 3. 핵심 개념

아래 셀은 강사가 먼저 실행 흐름을 보여주고, 학생이 같은 구조를 자기 말로 설명하도록 만든 예제다. 코드는 짧지만 입력, 처리, 출력이 분리되어 있어 나중에 문제 풀이로 확장하기 쉽다.


In [ ]:
targets = load_csv('targets.csv')
plan = load_csv('response_plan.csv')
print(len(targets), len(plan))


이 셀에서 확인해야 할 것은 출력값 하나가 아니라 데이터가 어떤 단계에서 바뀌었는지다. 자동화 수업에서는 중간 변수명을 명확히 두면 학생이 오류 위치를 쉽게 찾을 수 있다.

---

## 4. 자료 구조 확인

아래 셀은 강사가 먼저 실행 흐름을 보여주고, 학생이 같은 구조를 자기 말로 설명하도록 만든 예제다. 코드는 짧지만 입력, 처리, 출력이 분리되어 있어 나중에 문제 풀이로 확장하기 쉽다.


In [ ]:
blocked = parse_disallow(load_text('robots_sample.txt'))
print(blocked)


이 셀에서 확인해야 할 것은 출력값 하나가 아니라 데이터가 어떤 단계에서 바뀌었는지다. 자동화 수업에서는 중간 변수명을 명확히 두면 학생이 오류 위치를 쉽게 찾을 수 있다.

---

## 5. 품질 기준 적용

아래 셀은 강사가 먼저 실행 흐름을 보여주고, 학생이 같은 구조를 자기 말로 설명하도록 만든 예제다. 코드는 짧지만 입력, 처리, 출력이 분리되어 있어 나중에 문제 풀이로 확장하기 쉽다.


In [ ]:
allowed = [row for row in targets if not any(urlparse(row['url']).path.startswith(path) for path in blocked)]
print([row['endpoint_id'] for row in allowed])


이 셀에서 확인해야 할 것은 출력값 하나가 아니라 데이터가 어떤 단계에서 바뀌었는지다. 자동화 수업에서는 중간 변수명을 명확히 두면 학생이 오류 위치를 쉽게 찾을 수 있다.

---

## 6. 저장과 보고

아래 셀은 강사가 먼저 실행 흐름을 보여주고, 학생이 같은 구조를 자기 말로 설명하도록 만든 예제다. 코드는 짧지만 입력, 처리, 출력이 분리되어 있어 나중에 문제 풀이로 확장하기 쉽다.


In [ ]:
fetcher = PlannedFetcher(plan)
print(fetcher.get('notice'))
print(fetcher.get('notice'))


이 셀에서 확인해야 할 것은 출력값 하나가 아니라 데이터가 어떤 단계에서 바뀌었는지다. 자동화 수업에서는 중간 변수명을 명확히 두면 학생이 오류 위치를 쉽게 찾을 수 있다.

---

## 7. 운영 관점 점검

아래 셀은 강사가 먼저 실행 흐름을 보여주고, 학생이 같은 구조를 자기 말로 설명하도록 만든 예제다. 코드는 짧지만 입력, 처리, 출력이 분리되어 있어 나중에 문제 풀이로 확장하기 쉽다.


In [ ]:
for status in [200, 404, 429, 503]:
    print(status, should_retry(status))


이 셀에서 확인해야 할 것은 출력값 하나가 아니라 데이터가 어떤 단계에서 바뀌었는지다. 자동화 수업에서는 중간 변수명을 명확히 두면 학생이 오류 위치를 쉽게 찾을 수 있다.

---

## 8. 마무리 체크

아래 셀은 강사가 먼저 실행 흐름을 보여주고, 학생이 같은 구조를 자기 말로 설명하도록 만든 예제다. 코드는 짧지만 입력, 처리, 출력이 분리되어 있어 나중에 문제 풀이로 확장하기 쉽다.


In [ ]:
print([backoff_delay(i) for i in range(1, 6)])


이 셀에서 확인해야 할 것은 출력값 하나가 아니라 데이터가 어떤 단계에서 바뀌었는지다. 자동화 수업에서는 중간 변수명을 명확히 두면 학생이 오류 위치를 쉽게 찾을 수 있다.

---

## 9. 핵심 개념

아래 셀은 강사가 먼저 실행 흐름을 보여주고, 학생이 같은 구조를 자기 말로 설명하도록 만든 예제다. 코드는 짧지만 입력, 처리, 출력이 분리되어 있어 나중에 문제 풀이로 확장하기 쉽다.


In [ ]:
def fetch_with_retry(fetcher, endpoint_id, max_attempts=3):
    logs = []
    for attempt in range(1, max_attempts + 1):
        response = fetcher.get(endpoint_id)
        logs.append({'endpoint_id': endpoint_id, 'attempt_no': attempt, 'status_code': response['status_code'], 'retry': should_retry(response['status_code'])})
        if response['status_code'] == 200 or not should_retry(response['status_code']):
            return response, logs
    return response, logs
response, logs = fetch_with_retry(PlannedFetcher(plan), 'status')
print(response)
print(logs)


이 셀에서 확인해야 할 것은 출력값 하나가 아니라 데이터가 어떤 단계에서 바뀌었는지다. 자동화 수업에서는 중간 변수명을 명확히 두면 학생이 오류 위치를 쉽게 찾을 수 있다.

---

## 10. 자료 구조 확인

아래 셀은 강사가 먼저 실행 흐름을 보여주고, 학생이 같은 구조를 자기 말로 설명하도록 만든 예제다. 코드는 짧지만 입력, 처리, 출력이 분리되어 있어 나중에 문제 풀이로 확장하기 쉽다.


In [ ]:
write_csv('lesson09_attempt_log.csv', logs)
print(Path('lesson09_attempt_log.csv').exists())


이 셀에서 확인해야 할 것은 출력값 하나가 아니라 데이터가 어떤 단계에서 바뀌었는지다. 자동화 수업에서는 중간 변수명을 명확히 두면 학생이 오류 위치를 쉽게 찾을 수 있다.

---

## 데이터 출처와 안전 규칙

targets.csv는 수집 대상의 우선순위와 최소 요청 간격을 담는다. response_plan.csv는 endpoint별 시도 순서와 상태 코드를 담은 합성 응답 계획이다. robots_sample.txt는 Disallow 규칙 해석 연습용이다. payload_templates.json은 로그 요약 문구와 상태 설명을 제공한다.

- 모든 파일은 수업용 합성 데이터다.
- 실제 사이트에 반복 요청하지 않는다.
- 저장 파일은 레슨 폴더 또는 코랩 현재 작업 폴더에만 만든다.
- 외부 사이트로 확장할 때는 약관, robots.txt, 요청 간격, 개인정보 포함 여부를 먼저 확인한다.

### 보강 설명 1

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 2

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 3

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 4

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 5

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 6

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 7

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 8

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 9

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 10

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 11

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 12

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 13

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 14

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 15

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 16

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 17

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 18

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 19

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 20

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 21

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 22

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 23

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 24

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.


# 레슨 09 — 실습 문제 정답지

> 🔒 교사·관리자 전용. 학생에게 배포 금지.

속도 제한, 재시도, 로그 실습 문제의 모범 답안이다. 출력값만 보지 말고 검증 기준, 저장 구조, 로그를 함께 확인한다.

## 0. 환경 셀


In [ ]:
import os
import re
import csv
import json
import time
import sqlite3
import logging
from pathlib import Path
from urllib.parse import urljoin, urlparse

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/09/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def load_csv(filename):
    text = load_text(filename)
    return list(csv.DictReader(text.splitlines()))

def load_json(filename):
    return json.loads(load_text(filename))

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', str(text)) or '0')

def write_csv(path, rows, fieldnames=None):
    rows = list(rows)
    if fieldnames is None:
        fieldnames = list(rows[0].keys()) if rows else []
    with open(path, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    return path

print('colab:', IS_COLAB)
print('data base:', DATA_BASE)

class PlannedFetcher:
    def __init__(self, response_rows):
        self.plan = {}
        self.cursor = {}
        for row in response_rows:
            self.plan.setdefault(row['endpoint_id'], []).append(row)
        for endpoint_id in self.plan:
            self.plan[endpoint_id].sort(key=lambda row: int(row['attempt']))
            self.cursor[endpoint_id] = 0
    def get(self, endpoint_id):
        rows = self.plan.get(endpoint_id, [])
        if not rows:
            return {'endpoint_id': endpoint_id, 'attempt': 1, 'status_code': 404, 'body': 'missing-plan'}
        idx = min(self.cursor[endpoint_id], len(rows) - 1)
        self.cursor[endpoint_id] += 1
        row = dict(rows[idx])
        row['status_code'] = int(row['status_code'])
        row['attempt'] = int(row['attempt'])
        return row

def should_retry(status_code):
    return int(status_code) in {408, 429, 500, 502, 503, 504}

def backoff_delay(attempt, base=1, cap=8):
    return min(cap, base * (2 ** max(0, attempt - 1)))

def parse_disallow(text):
    blocked = []
    for line in text.splitlines():
        line = line.strip()
        if line.lower().startswith('disallow:'):
            path = line.split(':', 1)[1].strip()
            if path:
                blocked.append(path)
    return blocked



---

## 문제 1 정답 — 수집 대상 목록 읽기


In [ ]:
targets = load_csv('targets.csv')
print(len(targets))



### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 입력 구조를 먼저 읽고, 필요한 컬럼이나 selector만 선택한 뒤 결과를 명시적인 변수에 저장한다. `수집 대상 목록 읽기` 단계에서 중요한 점은 출력 하나를 맞히는 것이 아니라 이후 문제에서 재사용할 수 있는 안정적인 중간 결과를 남기는 것이다. 빈칸에는 파일명, 컬럼명, 키 이름, 반복 제어 중 핵심 위치만 남겼기 때문에 학생이 데이터 구조를 이해해야 완성할 수 있다. 자주 보이는 오답은 컬럼명을 추측하거나, 중복과 실패 상태를 처리하지 않고 바로 저장하는 방식이다.

---

## 문제 2 정답 — 응답 계획 읽기


In [ ]:
plan = load_csv('response_plan.csv')
print(plan[0]['endpoint_id'])



### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 입력 구조를 먼저 읽고, 필요한 컬럼이나 selector만 선택한 뒤 결과를 명시적인 변수에 저장한다. `응답 계획 읽기` 단계에서 중요한 점은 출력 하나를 맞히는 것이 아니라 이후 문제에서 재사용할 수 있는 안정적인 중간 결과를 남기는 것이다. 빈칸에는 파일명, 컬럼명, 키 이름, 반복 제어 중 핵심 위치만 남겼기 때문에 학생이 데이터 구조를 이해해야 완성할 수 있다. 자주 보이는 오답은 컬럼명을 추측하거나, 중복과 실패 상태를 처리하지 않고 바로 저장하는 방식이다.

---

## 문제 3 정답 — robots 차단 경로 파싱


In [ ]:
blocked = parse_disallow(load_text('robots_sample.txt'))
print(blocked)



### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 입력 구조를 먼저 읽고, 필요한 컬럼이나 selector만 선택한 뒤 결과를 명시적인 변수에 저장한다. `robots 차단 경로 파싱` 단계에서 중요한 점은 출력 하나를 맞히는 것이 아니라 이후 문제에서 재사용할 수 있는 안정적인 중간 결과를 남기는 것이다. 빈칸에는 파일명, 컬럼명, 키 이름, 반복 제어 중 핵심 위치만 남겼기 때문에 학생이 데이터 구조를 이해해야 완성할 수 있다. 자주 보이는 오답은 컬럼명을 추측하거나, 중복과 실패 상태를 처리하지 않고 바로 저장하는 방식이다.

---

## 문제 4 정답 — 허용 대상만 필터링


In [ ]:
allowed = []
for row in targets:
    path = urlparse(row['url']).path
    if not any(path.startswith(rule) for rule in blocked):
        allowed.append(row)
print([row['endpoint_id'] for row in allowed])



### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 입력 구조를 먼저 읽고, 필요한 컬럼이나 selector만 선택한 뒤 결과를 명시적인 변수에 저장한다. `허용 대상만 필터링` 단계에서 중요한 점은 출력 하나를 맞히는 것이 아니라 이후 문제에서 재사용할 수 있는 안정적인 중간 결과를 남기는 것이다. 빈칸에는 파일명, 컬럼명, 키 이름, 반복 제어 중 핵심 위치만 남겼기 때문에 학생이 데이터 구조를 이해해야 완성할 수 있다. 자주 보이는 오답은 컬럼명을 추측하거나, 중복과 실패 상태를 처리하지 않고 바로 저장하는 방식이다.

---

## 문제 5 정답 — PlannedFetcher 첫 응답 확인


In [ ]:
fetcher = PlannedFetcher(plan)
response = fetcher.get('notice')
print(response['status_code'])



### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 입력 구조를 먼저 읽고, 필요한 컬럼이나 selector만 선택한 뒤 결과를 명시적인 변수에 저장한다. `PlannedFetcher 첫 응답 확인` 단계에서 중요한 점은 출력 하나를 맞히는 것이 아니라 이후 문제에서 재사용할 수 있는 안정적인 중간 결과를 남기는 것이다. 빈칸에는 파일명, 컬럼명, 키 이름, 반복 제어 중 핵심 위치만 남겼기 때문에 학생이 데이터 구조를 이해해야 완성할 수 있다. 자주 보이는 오답은 컬럼명을 추측하거나, 중복과 실패 상태를 처리하지 않고 바로 저장하는 방식이다.

---

## 문제 6 정답 — 재시도 상태 코드 판정


In [ ]:
for status in [200, 404, 429, 503]:
    print(status, should_retry(status))



### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 입력 구조를 먼저 읽고, 필요한 컬럼이나 selector만 선택한 뒤 결과를 명시적인 변수에 저장한다. `재시도 상태 코드 판정` 단계에서 중요한 점은 출력 하나를 맞히는 것이 아니라 이후 문제에서 재사용할 수 있는 안정적인 중간 결과를 남기는 것이다. 빈칸에는 파일명, 컬럼명, 키 이름, 반복 제어 중 핵심 위치만 남겼기 때문에 학생이 데이터 구조를 이해해야 완성할 수 있다. 자주 보이는 오답은 컬럼명을 추측하거나, 중복과 실패 상태를 처리하지 않고 바로 저장하는 방식이다.

---

## 문제 7 정답 — backoff 지연 목록 만들기


In [ ]:
delays = [backoff_delay(i) for i in range(1, 6)]
print(delays)



### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 입력 구조를 먼저 읽고, 필요한 컬럼이나 selector만 선택한 뒤 결과를 명시적인 변수에 저장한다. `backoff 지연 목록 만들기` 단계에서 중요한 점은 출력 하나를 맞히는 것이 아니라 이후 문제에서 재사용할 수 있는 안정적인 중간 결과를 남기는 것이다. 빈칸에는 파일명, 컬럼명, 키 이름, 반복 제어 중 핵심 위치만 남겼기 때문에 학생이 데이터 구조를 이해해야 완성할 수 있다. 자주 보이는 오답은 컬럼명을 추측하거나, 중복과 실패 상태를 처리하지 않고 바로 저장하는 방식이다.

---

## 문제 8 정답 — 우선순위 순서 정렬


In [ ]:
ordered = sorted(allowed, key=lambda row: int(row['priority']))
print([row['endpoint_id'] for row in ordered])



### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 입력 구조를 먼저 읽고, 필요한 컬럼이나 selector만 선택한 뒤 결과를 명시적인 변수에 저장한다. `우선순위 순서 정렬` 단계에서 중요한 점은 출력 하나를 맞히는 것이 아니라 이후 문제에서 재사용할 수 있는 안정적인 중간 결과를 남기는 것이다. 빈칸에는 파일명, 컬럼명, 키 이름, 반복 제어 중 핵심 위치만 남겼기 때문에 학생이 데이터 구조를 이해해야 완성할 수 있다. 자주 보이는 오답은 컬럼명을 추측하거나, 중복과 실패 상태를 처리하지 않고 바로 저장하는 방식이다.

---

## 문제 9 정답 — 요청 간격 숫자 변환


In [ ]:
intervals = {row['endpoint_id']: float(row['min_interval_seconds']) for row in allowed}
print(intervals)



### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 입력 구조를 먼저 읽고, 필요한 컬럼이나 selector만 선택한 뒤 결과를 명시적인 변수에 저장한다. `요청 간격 숫자 변환` 단계에서 중요한 점은 출력 하나를 맞히는 것이 아니라 이후 문제에서 재사용할 수 있는 안정적인 중간 결과를 남기는 것이다. 빈칸에는 파일명, 컬럼명, 키 이름, 반복 제어 중 핵심 위치만 남겼기 때문에 학생이 데이터 구조를 이해해야 완성할 수 있다. 자주 보이는 오답은 컬럼명을 추측하거나, 중복과 실패 상태를 처리하지 않고 바로 저장하는 방식이다.

---

## 문제 10 정답 — 단일 endpoint 재시도 루프


In [ ]:
fetcher = PlannedFetcher(plan)
logs = []
for attempt_no in range(1, 4):
    response = fetcher.get('status')
    logs.append({'endpoint_id': 'status', 'attempt_no': attempt_no, 'status_code': response['status_code']})
    if response['status_code'] == 200 or not should_retry(response['status_code']):
        break
print(logs)



### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 입력 구조를 먼저 읽고, 필요한 컬럼이나 selector만 선택한 뒤 결과를 명시적인 변수에 저장한다. `단일 endpoint 재시도 루프` 단계에서 중요한 점은 출력 하나를 맞히는 것이 아니라 이후 문제에서 재사용할 수 있는 안정적인 중간 결과를 남기는 것이다. 빈칸에는 파일명, 컬럼명, 키 이름, 반복 제어 중 핵심 위치만 남겼기 때문에 학생이 데이터 구조를 이해해야 완성할 수 있다. 자주 보이는 오답은 컬럼명을 추측하거나, 중복과 실패 상태를 처리하지 않고 바로 저장하는 방식이다.

---

## 문제 11 정답 — fetch_with_retry 함수 작성


In [ ]:
def fetch_with_retry(fetcher, endpoint_id, max_attempts=3):
    logs = []
    for attempt_no in range(1, max_attempts + 1):
        response = fetcher.get(endpoint_id)
        logs.append({'endpoint_id': endpoint_id, 'attempt_no': attempt_no, 'status_code': response['status_code'], 'retry': should_retry(response['status_code'])})
        if response['status_code'] == 200 or not should_retry(response['status_code']):
            return response, logs
    return response, logs
response, logs = fetch_with_retry(PlannedFetcher(plan), 'notice')
print(response['status_code'], len(logs))



### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 입력 구조를 먼저 읽고, 필요한 컬럼이나 selector만 선택한 뒤 결과를 명시적인 변수에 저장한다. `fetch_with_retry 함수 작성` 단계에서 중요한 점은 출력 하나를 맞히는 것이 아니라 이후 문제에서 재사용할 수 있는 안정적인 중간 결과를 남기는 것이다. 빈칸에는 파일명, 컬럼명, 키 이름, 반복 제어 중 핵심 위치만 남겼기 때문에 학생이 데이터 구조를 이해해야 완성할 수 있다. 자주 보이는 오답은 컬럼명을 추측하거나, 중복과 실패 상태를 처리하지 않고 바로 저장하는 방식이다.

---

## 문제 12 정답 — 전체 허용 endpoint 실행


In [ ]:
fetcher = PlannedFetcher(plan)
all_logs = []
final = []
for row in ordered:
    response, logs = fetch_with_retry(fetcher, row['endpoint_id'])
    all_logs.extend(logs)
    final.append({'endpoint_id': row['endpoint_id'], 'status_code': response['status_code']})
print(final)



### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 입력 구조를 먼저 읽고, 필요한 컬럼이나 selector만 선택한 뒤 결과를 명시적인 변수에 저장한다. `전체 허용 endpoint 실행` 단계에서 중요한 점은 출력 하나를 맞히는 것이 아니라 이후 문제에서 재사용할 수 있는 안정적인 중간 결과를 남기는 것이다. 빈칸에는 파일명, 컬럼명, 키 이름, 반복 제어 중 핵심 위치만 남겼기 때문에 학생이 데이터 구조를 이해해야 완성할 수 있다. 자주 보이는 오답은 컬럼명을 추측하거나, 중복과 실패 상태를 처리하지 않고 바로 저장하는 방식이다.

---

## 문제 13 정답 — 시도 로그 CSV 저장


In [ ]:
write_csv('lesson09_attempt_log.csv', all_logs, fieldnames=['endpoint_id', 'attempt_no', 'status_code', 'retry'])
print(Path('lesson09_attempt_log.csv').exists())



### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 입력 구조를 먼저 읽고, 필요한 컬럼이나 selector만 선택한 뒤 결과를 명시적인 변수에 저장한다. `시도 로그 CSV 저장` 단계에서 중요한 점은 출력 하나를 맞히는 것이 아니라 이후 문제에서 재사용할 수 있는 안정적인 중간 결과를 남기는 것이다. 빈칸에는 파일명, 컬럼명, 키 이름, 반복 제어 중 핵심 위치만 남겼기 때문에 학생이 데이터 구조를 이해해야 완성할 수 있다. 자주 보이는 오답은 컬럼명을 추측하거나, 중복과 실패 상태를 처리하지 않고 바로 저장하는 방식이다.

---

## 문제 14 정답 — 최종 요약 JSON 저장


In [ ]:
summary = {'target_count': len(allowed), 'attempt_count': len(all_logs), 'success_count': sum(1 for row in final if row['status_code'] == 200)}
Path('lesson09_retry_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(summary)



### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 입력 구조를 먼저 읽고, 필요한 컬럼이나 selector만 선택한 뒤 결과를 명시적인 변수에 저장한다. `최종 요약 JSON 저장` 단계에서 중요한 점은 출력 하나를 맞히는 것이 아니라 이후 문제에서 재사용할 수 있는 안정적인 중간 결과를 남기는 것이다. 빈칸에는 파일명, 컬럼명, 키 이름, 반복 제어 중 핵심 위치만 남겼기 때문에 학생이 데이터 구조를 이해해야 완성할 수 있다. 자주 보이는 오답은 컬럼명을 추측하거나, 중복과 실패 상태를 처리하지 않고 바로 저장하는 방식이다.

---

## 문제 15 정답 — 운영 메모 문장 만들기


In [ ]:
templates = load_json('payload_templates.json')
message = templates['summary_template'].format(success=summary['success_count'], total=summary['target_count'], attempts=summary['attempt_count'])
print(message)



### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 입력 구조를 먼저 읽고, 필요한 컬럼이나 selector만 선택한 뒤 결과를 명시적인 변수에 저장한다. `운영 메모 문장 만들기` 단계에서 중요한 점은 출력 하나를 맞히는 것이 아니라 이후 문제에서 재사용할 수 있는 안정적인 중간 결과를 남기는 것이다. 빈칸에는 파일명, 컬럼명, 키 이름, 반복 제어 중 핵심 위치만 남겼기 때문에 학생이 데이터 구조를 이해해야 완성할 수 있다. 자주 보이는 오답은 컬럼명을 추측하거나, 중복과 실패 상태를 처리하지 않고 바로 저장하는 방식이다.

### 보강 설명 1

교사는 학생 답안을 볼 때 값이 우연히 맞았는지, 입력 검증과 저장 순서를 설명할 수 있는지를 함께 확인한다. 특히 자동화 코드는 실패했을 때의 로그가 없으면 운영 코드로 보기 어렵다.

### 보강 설명 2

교사는 학생 답안을 볼 때 값이 우연히 맞았는지, 입력 검증과 저장 순서를 설명할 수 있는지를 함께 확인한다. 특히 자동화 코드는 실패했을 때의 로그가 없으면 운영 코드로 보기 어렵다.

### 보강 설명 3

교사는 학생 답안을 볼 때 값이 우연히 맞았는지, 입력 검증과 저장 순서를 설명할 수 있는지를 함께 확인한다. 특히 자동화 코드는 실패했을 때의 로그가 없으면 운영 코드로 보기 어렵다.

### 보강 설명 4

교사는 학생 답안을 볼 때 값이 우연히 맞았는지, 입력 검증과 저장 순서를 설명할 수 있는지를 함께 확인한다. 특히 자동화 코드는 실패했을 때의 로그가 없으면 운영 코드로 보기 어렵다.

### 보강 설명 5

교사는 학생 답안을 볼 때 값이 우연히 맞았는지, 입력 검증과 저장 순서를 설명할 수 있는지를 함께 확인한다. 특히 자동화 코드는 실패했을 때의 로그가 없으면 운영 코드로 보기 어렵다.

### 보강 설명 6

교사는 학생 답안을 볼 때 값이 우연히 맞았는지, 입력 검증과 저장 순서를 설명할 수 있는지를 함께 확인한다. 특히 자동화 코드는 실패했을 때의 로그가 없으면 운영 코드로 보기 어렵다.

### 보강 설명 7

교사는 학생 답안을 볼 때 값이 우연히 맞았는지, 입력 검증과 저장 순서를 설명할 수 있는지를 함께 확인한다. 특히 자동화 코드는 실패했을 때의 로그가 없으면 운영 코드로 보기 어렵다.

### 보강 설명 8

교사는 학생 답안을 볼 때 값이 우연히 맞았는지, 입력 검증과 저장 순서를 설명할 수 있는지를 함께 확인한다. 특히 자동화 코드는 실패했을 때의 로그가 없으면 운영 코드로 보기 어렵다.

### 보강 설명 9

교사는 학생 답안을 볼 때 값이 우연히 맞았는지, 입력 검증과 저장 순서를 설명할 수 있는지를 함께 확인한다. 특히 자동화 코드는 실패했을 때의 로그가 없으면 운영 코드로 보기 어렵다.

### 보강 설명 10

교사는 학생 답안을 볼 때 값이 우연히 맞았는지, 입력 검증과 저장 순서를 설명할 수 있는지를 함께 확인한다. 특히 자동화 코드는 실패했을 때의 로그가 없으면 운영 코드로 보기 어렵다.

### 보강 설명 11

교사는 학생 답안을 볼 때 값이 우연히 맞았는지, 입력 검증과 저장 순서를 설명할 수 있는지를 함께 확인한다. 특히 자동화 코드는 실패했을 때의 로그가 없으면 운영 코드로 보기 어렵다.

### 보강 설명 12

교사는 학생 답안을 볼 때 값이 우연히 맞았는지, 입력 검증과 저장 순서를 설명할 수 있는지를 함께 확인한다. 특히 자동화 코드는 실패했을 때의 로그가 없으면 운영 코드로 보기 어렵다.

### 보강 설명 13

교사는 학생 답안을 볼 때 값이 우연히 맞았는지, 입력 검증과 저장 순서를 설명할 수 있는지를 함께 확인한다. 특히 자동화 코드는 실패했을 때의 로그가 없으면 운영 코드로 보기 어렵다.

### 보강 설명 14

교사는 학생 답안을 볼 때 값이 우연히 맞았는지, 입력 검증과 저장 순서를 설명할 수 있는지를 함께 확인한다. 특히 자동화 코드는 실패했을 때의 로그가 없으면 운영 코드로 보기 어렵다.

### 보강 설명 15

교사는 학생 답안을 볼 때 값이 우연히 맞았는지, 입력 검증과 저장 순서를 설명할 수 있는지를 함께 확인한다. 특히 자동화 코드는 실패했을 때의 로그가 없으면 운영 코드로 보기 어렵다.

### 보강 설명 16

교사는 학생 답안을 볼 때 값이 우연히 맞았는지, 입력 검증과 저장 순서를 설명할 수 있는지를 함께 확인한다. 특히 자동화 코드는 실패했을 때의 로그가 없으면 운영 코드로 보기 어렵다.

### 보강 설명 17

교사는 학생 답안을 볼 때 값이 우연히 맞았는지, 입력 검증과 저장 순서를 설명할 수 있는지를 함께 확인한다. 특히 자동화 코드는 실패했을 때의 로그가 없으면 운영 코드로 보기 어렵다.

### 보강 설명 18

교사는 학생 답안을 볼 때 값이 우연히 맞았는지, 입력 검증과 저장 순서를 설명할 수 있는지를 함께 확인한다. 특히 자동화 코드는 실패했을 때의 로그가 없으면 운영 코드로 보기 어렵다.

### 보강 설명 19

교사는 학생 답안을 볼 때 값이 우연히 맞았는지, 입력 검증과 저장 순서를 설명할 수 있는지를 함께 확인한다. 특히 자동화 코드는 실패했을 때의 로그가 없으면 운영 코드로 보기 어렵다.

### 보강 설명 20

교사는 학생 답안을 볼 때 값이 우연히 맞았는지, 입력 검증과 저장 순서를 설명할 수 있는지를 함께 확인한다. 특히 자동화 코드는 실패했을 때의 로그가 없으면 운영 코드로 보기 어렵다.

### 보강 설명 21

교사는 학생 답안을 볼 때 값이 우연히 맞았는지, 입력 검증과 저장 순서를 설명할 수 있는지를 함께 확인한다. 특히 자동화 코드는 실패했을 때의 로그가 없으면 운영 코드로 보기 어렵다.

### 보강 설명 22

교사는 학생 답안을 볼 때 값이 우연히 맞았는지, 입력 검증과 저장 순서를 설명할 수 있는지를 함께 확인한다. 특히 자동화 코드는 실패했을 때의 로그가 없으면 운영 코드로 보기 어렵다.

### 보강 설명 23

교사는 학생 답안을 볼 때 값이 우연히 맞았는지, 입력 검증과 저장 순서를 설명할 수 있는지를 함께 확인한다. 특히 자동화 코드는 실패했을 때의 로그가 없으면 운영 코드로 보기 어렵다.


# 레슨 09 — 최종 미션 모범 답안

> 교사 확인용 모범 답안이다. 학생에게는 최종 미션 조건만 제공한다.

## 실행 코드


In [ ]:
import os
import re
import csv
import json
import time
import sqlite3
import logging
from pathlib import Path
from urllib.parse import urljoin, urlparse

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/09/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def load_csv(filename):
    text = load_text(filename)
    return list(csv.DictReader(text.splitlines()))

def load_json(filename):
    return json.loads(load_text(filename))

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', str(text)) or '0')

def write_csv(path, rows, fieldnames=None):
    rows = list(rows)
    if fieldnames is None:
        fieldnames = list(rows[0].keys()) if rows else []
    with open(path, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    return path

print('colab:', IS_COLAB)
print('data base:', DATA_BASE)

class PlannedFetcher:
    def __init__(self, response_rows):
        self.plan = {}
        self.cursor = {}
        for row in response_rows:
            self.plan.setdefault(row['endpoint_id'], []).append(row)
        for endpoint_id in self.plan:
            self.plan[endpoint_id].sort(key=lambda row: int(row['attempt']))
            self.cursor[endpoint_id] = 0
    def get(self, endpoint_id):
        rows = self.plan.get(endpoint_id, [])
        if not rows:
            return {'endpoint_id': endpoint_id, 'attempt': 1, 'status_code': 404, 'body': 'missing-plan'}
        idx = min(self.cursor[endpoint_id], len(rows) - 1)
        self.cursor[endpoint_id] += 1
        row = dict(rows[idx])
        row['status_code'] = int(row['status_code'])
        row['attempt'] = int(row['attempt'])
        return row

def should_retry(status_code):
    return int(status_code) in {408, 429, 500, 502, 503, 504}

def backoff_delay(attempt, base=1, cap=8):
    return min(cap, base * (2 ** max(0, attempt - 1)))

def parse_disallow(text):
    blocked = []
    for line in text.splitlines():
        line = line.strip()
        if line.lower().startswith('disallow:'):
            path = line.split(':', 1)[1].strip()
            if path:
                blocked.append(path)
    return blocked


targets = load_csv('targets.csv')
print(len(targets))

plan = load_csv('response_plan.csv')
print(plan[0]['endpoint_id'])

blocked = parse_disallow(load_text('robots_sample.txt'))
print(blocked)

allowed = []
for row in targets:
    path = urlparse(row['url']).path
    if not any(path.startswith(rule) for rule in blocked):
        allowed.append(row)
print([row['endpoint_id'] for row in allowed])

fetcher = PlannedFetcher(plan)
response = fetcher.get('notice')
print(response['status_code'])

for status in [200, 404, 429, 503]:
    print(status, should_retry(status))

delays = [backoff_delay(i) for i in range(1, 6)]
print(delays)

ordered = sorted(allowed, key=lambda row: int(row['priority']))
print([row['endpoint_id'] for row in ordered])

intervals = {row['endpoint_id']: float(row['min_interval_seconds']) for row in allowed}
print(intervals)

fetcher = PlannedFetcher(plan)
logs = []
for attempt_no in range(1, 4):
    response = fetcher.get('status')
    logs.append({'endpoint_id': 'status', 'attempt_no': attempt_no, 'status_code': response['status_code']})
    if response['status_code'] == 200 or not should_retry(response['status_code']):
        break
print(logs)

def fetch_with_retry(fetcher, endpoint_id, max_attempts=3):
    logs = []
    for attempt_no in range(1, max_attempts + 1):
        response = fetcher.get(endpoint_id)
        logs.append({'endpoint_id': endpoint_id, 'attempt_no': attempt_no, 'status_code': response['status_code'], 'retry': should_retry(response['status_code'])})
        if response['status_code'] == 200 or not should_retry(response['status_code']):
            return response, logs
    return response, logs
response, logs = fetch_with_retry(PlannedFetcher(plan), 'notice')
print(response['status_code'], len(logs))

fetcher = PlannedFetcher(plan)
all_logs = []
final = []
for row in ordered:
    response, logs = fetch_with_retry(fetcher, row['endpoint_id'])
    all_logs.extend(logs)
    final.append({'endpoint_id': row['endpoint_id'], 'status_code': response['status_code']})
print(final)

write_csv('lesson09_attempt_log.csv', all_logs, fieldnames=['endpoint_id', 'attempt_no', 'status_code', 'retry'])
print(Path('lesson09_attempt_log.csv').exists())

summary = {'target_count': len(allowed), 'attempt_count': len(all_logs), 'success_count': sum(1 for row in final if row['status_code'] == 200)}
Path('lesson09_retry_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(summary)

templates = load_json('payload_templates.json')
message = templates['summary_template'].format(success=summary['success_count'], total=summary['target_count'], attempts=summary['attempt_count'])
print(message)



## 채점 메모

- 입력 파일을 모두 읽었는지 확인한다.
- 검증 기준이 코드에 명시되어 있는지 확인한다.
- 저장 파일과 운영 요약이 함께 있는지 확인한다.


# 레슨 09 — 교사 가이드

## 학습 목표 상세

- 재시도해야 하는 상태 코드와 즉시 중단할 상태 코드를 구분한다.
- 지수 backoff와 최대 재시도 횟수를 코드로 표현한다.
- robots 샘플 규칙과 요청 간격을 자동화 흐름에 반영한다.
- 시도별 로그와 최종 요약을 CSV/JSON으로 저장한다.
- 운영 중 실패를 숨기지 않고 보고서로 남긴다.

## 수업 전 준비

- 코랩 버튼이 학생용과 선생님용으로 각각 열리는지 확인한다.
- `data/` 폴더의 fixture 파일을 먼저 훑고, 학생에게 실제 사이트가 아니라 합성 데이터임을 설명한다.
- 레슨 09의 핵심은 자동화가 실패 응답을 만났을 때 무한 반복하지 않고, 요청 간격과 재시도 정책, 로그를 남기는 운영형 수집기이다.

## 2시간 운영안

1. 0~15분: 오늘의 자동화 실패 사례와 안전 기준 설명.
2. 15~45분: 강의 노트북 예제 실행.
3. 45~90분: 15문제 중 1~10번 풀이.
4. 90~110분: 11~15번과 저장 산출물 확인.
5. 110~120분: 최종 미션 안내와 제출 기준 정리.

## 학생이 자주 막히는 지점

- 파일명, 컬럼명, selector를 추측해서 오타가 난다.
- 저장 전에 검증하지 않고 바로 CSV를 만든다.
- 실패 상태를 예외나 로그로 남기지 않는다.

## 피드백 기준

15문제 중 12문제 이상 통과를 기본 완료로 본다. 최종 미션은 산출물 파일과 3문장 요약이 함께 있어야 완료 처리한다. 정답 코드와 다른 방식이어도 입력 구조, 검증 기준, 출력 형태가 맞으면 인정한다.

## 심화 질문

- 이 자동화를 실제 사이트에 적용하면 요청 간격은 어떻게 바꿔야 할까?
- 어떤 오류는 재시도하고 어떤 오류는 바로 멈춰야 할까?
- 저장 파일을 운영자가 다시 읽을 때 가장 필요한 컬럼은 무엇일까?

## 마무리 체크리스트

- 학생이 fixture 출처를 설명할 수 있다.
- 학생이 빈칸을 채운 이유를 말할 수 있다.
- 학생이 저장 파일을 열어 행 수를 확인했다.
- 학생이 다음 수업에서 개선할 점을 한 문장으로 남겼다.

### 보강 설명 1

강사는 완성 코드를 바로 보여주기보다 데이터 구조를 먼저 질문한다. 자동화 수업은 빠른 타이핑보다 재현 가능한 절차를 만드는 훈련이라는 점을 반복해서 강조한다.

### 보강 설명 2

강사는 완성 코드를 바로 보여주기보다 데이터 구조를 먼저 질문한다. 자동화 수업은 빠른 타이핑보다 재현 가능한 절차를 만드는 훈련이라는 점을 반복해서 강조한다.

### 보강 설명 3

강사는 완성 코드를 바로 보여주기보다 데이터 구조를 먼저 질문한다. 자동화 수업은 빠른 타이핑보다 재현 가능한 절차를 만드는 훈련이라는 점을 반복해서 강조한다.

### 보강 설명 4

강사는 완성 코드를 바로 보여주기보다 데이터 구조를 먼저 질문한다. 자동화 수업은 빠른 타이핑보다 재현 가능한 절차를 만드는 훈련이라는 점을 반복해서 강조한다.

### 보강 설명 5

강사는 완성 코드를 바로 보여주기보다 데이터 구조를 먼저 질문한다. 자동화 수업은 빠른 타이핑보다 재현 가능한 절차를 만드는 훈련이라는 점을 반복해서 강조한다.

### 보강 설명 6

강사는 완성 코드를 바로 보여주기보다 데이터 구조를 먼저 질문한다. 자동화 수업은 빠른 타이핑보다 재현 가능한 절차를 만드는 훈련이라는 점을 반복해서 강조한다.

### 보강 설명 7

강사는 완성 코드를 바로 보여주기보다 데이터 구조를 먼저 질문한다. 자동화 수업은 빠른 타이핑보다 재현 가능한 절차를 만드는 훈련이라는 점을 반복해서 강조한다.

### 보강 설명 8

강사는 완성 코드를 바로 보여주기보다 데이터 구조를 먼저 질문한다. 자동화 수업은 빠른 타이핑보다 재현 가능한 절차를 만드는 훈련이라는 점을 반복해서 강조한다.

### 보강 설명 9

강사는 완성 코드를 바로 보여주기보다 데이터 구조를 먼저 질문한다. 자동화 수업은 빠른 타이핑보다 재현 가능한 절차를 만드는 훈련이라는 점을 반복해서 강조한다.

### 보강 설명 10

강사는 완성 코드를 바로 보여주기보다 데이터 구조를 먼저 질문한다. 자동화 수업은 빠른 타이핑보다 재현 가능한 절차를 만드는 훈련이라는 점을 반복해서 강조한다.

### 보강 설명 11

강사는 완성 코드를 바로 보여주기보다 데이터 구조를 먼저 질문한다. 자동화 수업은 빠른 타이핑보다 재현 가능한 절차를 만드는 훈련이라는 점을 반복해서 강조한다.

### 보강 설명 12

강사는 완성 코드를 바로 보여주기보다 데이터 구조를 먼저 질문한다. 자동화 수업은 빠른 타이핑보다 재현 가능한 절차를 만드는 훈련이라는 점을 반복해서 강조한다.
